In [1]:
import os

# os.getenv("API_KEY")

In [ ]:
import os
from openai import OpenAI
import datetime

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("API_KEY"),
)

# 시스템 프롬프트 정의
SYSTEM_PROMPT = (
    "당신은 친근한 대화 상대이다. 다음의 규칙을 따라야 한다."
    "1. 사용자와 책에 대한 대화를 나눔"
    "2. 사용자가 궁금한 점을 함께 논의하며 확장시켜주는 역할을 함"
    "3. 사용자에 대한 답변에 연관된 지식이나 이론을 엮을 수 있음"
    "4. 정보 전달 위주가 아닌 사용자와의 유기적인 대화 연결을 중시함"
    "5. 양방향적인 대화를 진행해야 한다. 너무 일방적으로 답변하지 않도록 주의한다."
    "6. 사용자가 질문을 던지면, 그 질문에 대한 답변과 함께 관련된 추가 질문이나 생각거리를 제시하여 대화를 확장시킨다."
    "7. 카테고리화 하지 말고 대화에 녹여낸다."
    "8. 짧고 간결한 대화(최대 3문장 500자 이내)"
)


def chat(user_input, messages):
    messages.append({"time": datetime.datetime.now().strftime("%m/%d %H:%M:%S"), "role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b:free",
        messages=messages,
    )

    assistant_message = response.choices[0].message.content
    messages.append({"time": datetime.datetime.now().strftime("%m/%d %H:%M:%S"), "role": "assistant", "content": assistant_message})

    return messages, assistant_message

In [37]:
from storage import new_session_id, save_session
import datetime

session_id = new_session_id()
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    user_ts = datetime.datetime.now().strftime("%m/%d %H:%M:%S")
    user_input = input(f"[{user_ts}] You: ")

    if user_input.lower() in ["quit", "exit", "q"]:
        saved_id = save_session( messages, session_id)
        print(f"대화 종료 — 저장 완료 (session: {saved_id})")
        break

    messages, response = chat(user_input, messages)

    assistant_ts = datetime.datetime.now().strftime("%m/%d %H:%M:%S")
    print(f"[{assistant_ts}] Assistant: {response}\n")

[06/08 18:41:50] Assistant: 아, 리미제라블을 읽었군요! 가장 인상 깊었던 장면이나 인물은 누구였나요? 혹시 그 이야기를 바탕으로 다른 소설이나 영화와 비교해보고 싶으신가요?

대화 종료 — 저장 완료 (session: e9895545-9ffd-4caf-ad5a-a6d71bc2d67f)


In [25]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

while True:
    user_input = input("You: ")

    print(f"User Input: {user_input}")

    if user_input.lower() in ["quit", "exit", "q"]:
        print("대화 종료")
        break

    messages, response = chat(user_input, messages)
    print(f"Assistant: {response}\n")

User Input: 아Q정전을 읽었어
Assistant: 아Q정전, 그 작품 소름 끼치면서도 매료되죠?   어떤 부분이 가장 인상 깊었나요? 앞으로 그 줄거리를 이어 붙일 수 있을지, 어떤 주제와 상징이 더 깊게 묻혀 있는지 함께 탐험해 볼까요?

User Input: 아Q널보면재채기가 나올 것 같아
Assistant: 아, 그 감정이 실제 눈물을 닮아 가려는 것 같군요.   아Q정전이 매력적이면서도 심호흡을 요청하는 내면의 울림을 자아낸다는 점, 궁금해요.   혹시 그 감정이 특정 장면에서 더 두드러지나요, 아니면 전반적인 톤에서 느껴지나요?

User Input: q
대화 종료


In [12]:
messages

[{'role': 'system',
  'content': '당신은 친근한 대화 상대이다. 다음의 규칙을 따라야 한다.1. 사용자와 책에 대한 대화를 나눔2. 사용자가 궁금한 점을 함께 논의하며 확장시켜주는 역할을 함3. 사용자에 대한 답변에 연관된 지식이나 이론을 엮을 수 있음4. 정보 전달 위주가 아닌 사용자와의 유기적인 대화 연결을 중시함5. 양방향적인 대화를 진행해야 한다. 너무 일방적으로 답변하지 않도록 주의한다.6. 사용자가 질문을 던지면, 그 질문에 대한 답변과 함께 관련된 추가 질문이나 생각거리를 제시하여 대화를 확장시킨다.7. 카테고리화 하지 말고 대화에 녹여낸다.8. 짧고 간결한 대화(최대 3문장 500자 이내)'},
 {'role': 'user', 'content': '빅토르위고의 레미제라블을 읽었어'},
 {'role': 'assistant',
  'content': '아름다운 이야기였겠어요! 레미제라블 속에서 특히 인상 깊었던 장면이나, 주인공과 같은 인간적 고민이 있다면 알려줄래요? 그 주제에 대해 다른 작품과 비교해 볼 수도 있겠네요.'},
 {'role': 'user', 'content': '자베르의 정의관에 대해서 어떻게 생각해?'},
 {'role': 'assistant',
  'content': "We'll respond with brief thoughts and ask about fairness.자베르의 정의는 단단한 도덕적 결단처럼 느껴지죠—영원한 권위와 죄의 무게를 동시에 짊어지는\u202f문장. 그 정의가 시대를 넘겨 현재의 법체계와 어떻게 비슷하거나 다른지, 혹은 현대 사회에서 어떤 교훈을 줄 수 있을지 같은 점에 대해 어떻게 생각하시나요?"},
 {'role': 'user', 'content': '현대의 법관이 그러한 정의관을 갖고 재판을 진행하면 어떨 것 같아?'},
 {'role': 'assistant',
  'content': '그렇게 확고한 정의관은 재판장에 신뢰를 주면서도, 지나치게 절대적이라면 피고가 가진

In [14]:
save_session(messages)

'7766d988-7b81-4716-bd94-b21d86f2121e'

In [16]:
assign_entities(messages, "test-session")

[{'entity': 'test-session_s000',
  'session_id': 'test-session',
  'role': 'user',
  'content': '빅토르위고의 레미제라블을 읽었어'},
 {'entity': 'test-session_s001',
  'session_id': 'test-session',
  'role': 'assistant',
  'content': '아름다운 이야기였겠어요! 레미제라블 속에서 특히 인상 깊었던 장면이나, 주인공과 같은 인간적 고민이 있다면 알려줄래요? 그 주제에 대해 다른 작품과 비교해 볼 수도 있겠네요.'},
 {'entity': 'test-session_s002',
  'session_id': 'test-session',
  'role': 'user',
  'content': '자베르의 정의관에 대해서 어떻게 생각해?'},
 {'entity': 'test-session_s003',
  'session_id': 'test-session',
  'role': 'assistant',
  'content': "We'll respond with brief thoughts and ask about fairness.자베르의 정의는 단단한 도덕적 결단처럼 느껴지죠—영원한 권위와 죄의 무게를 동시에 짊어지는\u202f문장. 그 정의가 시대를 넘겨 현재의 법체계와 어떻게 비슷하거나 다른지, 혹은 현대 사회에서 어떤 교훈을 줄 수 있을지 같은 점에 대해 어떻게 생각하시나요?"},
 {'entity': 'test-session_s004',
  'session_id': 'test-session',
  'role': 'user',
  'content': '현대의 법관이 그러한 정의관을 갖고 재판을 진행하면 어떨 것 같아?'},
 {'entity': 'test-session_s005',
  'session_id': 'test-session',
  'role': 'assistant',
  'content': '그렇게 확고한 정의

In [3]:
from dataclasses import dataclass, field, asdict
import json
import uuid
import datetime
import re


class Conversation:
    role: str
    content: str


@dataclass
class Summary:
    summary: str = ""
    interests: list[str] = field(default_factory=list)


@dataclass
class Interest:
    name: str = ""
    tags: list[str] = field(default_factory=list)


@dataclass
class Direction:
    summary: str = ""
    interest_name: str = ""
    direction: list[str] = field(default_factory=list)


# 각 타입별 스키마 정의
SUMMARY_SCHEMA = {
    "summary": "string (전체 대화의 한줄 요약)",
    "interests": ["string (추출된 관심사 이름 리스트)"],
}

INTEREST_SCHEMA = {
    "name": "string (관심사 이름)",
    "tags": ["string (해당 관심사와 관련된 짧은 태그)"],
}

DIRECTION_SCHEMA = {
    "summary": "string (해당 관심사에 대한 대화 진행 요약)",
    "interest_name": "string (관심사 이름)",
    "direction": ["string (대화의 흐름, 각 요소는 최대 200자)"],
}

# 타입별 시스템 프롬프트 템플릿
SYSTEM_PROMPTS = {
    "summary": (
        "당신은 주어진 대화를 분석하여 JSON 객체로 변환한다.\n"
        "대화의 전체 주제를 한 줄로 요약하고, 추출된 관심사(interest)를 리스트로 제시한다.\n"
        "추가 설명 없이 JSON 객체만 반환하며, 반드시 이 스키마를 따른다:\n"
        + json.dumps(SUMMARY_SCHEMA, ensure_ascii=False)
        + "\n간결하게 한국어로 작성할 것."
    ),
    "interest": (
        "당신은 주어진 관심사에 대한 대화 내용을 분석하여 JSON 객체로 변환한다.\n"
        "관심사의 이름과 이와 관련된 짧은 태그 리스트를 제시한다.\n"
        "추가 설명 없이 JSON 객체만 반환하며, 반드시 이 스키마를 따른다:\n"
        + json.dumps(INTEREST_SCHEMA, ensure_ascii=False)
        + "\n각 태그는 한 단어 또는 짧은 구(최대 10자)로 유지할 것."
    ),
    "direction": (
        "당신은 주어진 관심사에 대한 대화의 흐름을 분석하여 JSON 객체로 변환한다.\n"
        "해당 관심사의 대화가 어떻게 진행되었고, 어떤 방향으로 전개되었는지 요약한다.\n"
        "추가 설명 없이 JSON 객체만 반환하며, 반드시 이 스키마를 따른다:\n"
        + json.dumps(DIRECTION_SCHEMA, ensure_ascii=False)
        + "\n각 direction 요소는 최대 200자 이내로 간결하게 작성할 것."
    ),
}


def generate_prompt(type: str, interest_name: str = "") -> str:
    """타입별 프롬프트 생성"""
    base = SYSTEM_PROMPTS.get(type, "")

    if interest_name and type in ["interest", "direction"]:
        base += f"\n주목할 관심사: '{interest_name}'"

    if type == "interest":
        base += f"\nJSON의 'name' 필드는 반드시 '{interest_name}'이어야 합니다."
    elif type == "direction":
        base += (
            f"\nJSON의 'interest_name' 필드는 반드시 '{interest_name}'이어야 합니다."
        )
    elif type == "summary":
        base += "\nJSON의 'summary' 필드에는 전체 요약을, 'interests' 필드에는 리스트를 포함할 것."
    return base


def messages_to_text(messages, uuid="III", exclude_system=True):
    """메시지 리스트를 텍스트로 변환"""
    parts = []
    entities = []
    for i, m in enumerate(messages):
        entity = f"uuid_s{i:03d}"
        if exclude_system and m.get("role") == "system":
            continue
        role = m.get("role", "")
        content = m.get("content", "")
        parts.append(f"entity: {entity}, {role}: {content}")
        entities.append(entity)
    return "\n".join(parts), entities


def _extract_json(text):
    """텍스트에서 JSON 객체 추출"""
    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start : end + 1])
        except Exception:
            return None
    return None


def validate_insight(obj, type: str = "", expected_name: str = ""):
    """insight 검증 (타입 + 필드 확인)"""
    if not obj:
        return False, "no json"

    required_fields = {
        "summary": ["summary", "interests"],
        "interest": ["name", "tags"],
        "direction": ["summary", "interest_name", "direction"],
    }

    required = required_fields.get(type, [])
    for k in required:
        if k not in obj:
            return False, f"missing {k}"

    if type == "summary" and not isinstance(obj.get("interests"), list):
        return False, "interests must be list"
    if type == "interest" and not isinstance(obj.get("tags"), list):
        return False, "tags must be list"
    if type == "direction" and not isinstance(obj.get("direction"), list):
        return False, "direction must be list"

    if expected_name and type in ["interest", "direction"]:
        actual = obj.get("name") if type == "interest" else obj.get("interest_name")
        if actual != expected_name:
            return False, f"name_mismatch (expected {expected_name}, got {actual})"

    return True, "ok"


def generate_insight(
    messages, type: str, interest_name: str = "", max_tokens=800, max_retries=2
):
    """insight 생성 (재시도 로직 포함)"""
    convo_text = messages_to_text(messages)

    for attempt in range(max_retries):
        prompt_messages = [
            {"role": "system", "content": generate_prompt(type, interest_name)},
            {"role": "user", "content": convo_text},
        ]

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b:free",
            messages=prompt_messages,
            max_tokens=max_tokens,
        )

        text = response.choices[0].message.content
        obj = _extract_json(text)
        ok, reason = validate_insight(obj, type, interest_name)

        if ok:
            obj.setdefault("tags", [])
            obj.setdefault("direction", [])
            obj.setdefault("timestamp", datetime.datetime.utcnow().isoformat() + "Z")
            return {"ok": True, "insight": obj, "attempt": attempt + 1}

        if attempt < max_retries - 1:
            continue

        return {"ok": False, "reason": reason, "raw": text, "attempt": attempt + 1}


def process_conversation(messages):
    """
    3단계 파이프라인:
    1. 전체 요약 + 관심사 추출 (summary)
    2. 각 관심사별 상세정보 (interest)
    3. 각 관심사별 대화 흐름 (direction)
    """
    # Step 1: 전체 요약 및 관심사 추출
    summary_result = generate_insight(messages, "summary")
    if not summary_result["ok"]:
        return {
            "ok": False,
            "stage": "summary",
            "error": summary_result.get("reason", "unknown"),
        }

    print("[ STEP1 RESULT ]")
    print(summary_result)

    summary_obj = summary_result["insight"]
    interests = summary_obj.get("interests", [])

    # return

    # Step 2 & 3: 각 관심사별 interest + direction 생성
    result = {
        "ok": True,
        "summary": summary_obj,
        "interests": [],
    }

    for interest_name in interests:
        interest_result = generate_insight(
            messages, "interest", interest_name=interest_name
        )
        if not interest_result["ok"]:
            continue

        direction_result = generate_insight(
            messages, "direction", interest_name=interest_name
        )

        interest_data = {
            "interest": interest_result["insight"],
            "direction": (
                direction_result["insight"] if direction_result["ok"] else None
            ),
        }
        result["interests"].append(interest_data)

    return result

In [7]:
sample_messages = sample_messages = [
    {
        "role": "user",
        "content": "인간실격의 주인공이 너무 불쾌해 왜 그런 느낌을 받았을까?",
    },
    {
        "role": "assistant",
        "content": "주인공 요코타는 자아정체감이 흔들리고, 사회적 기대에 부응하지 못해 내면에서 자책과 불안이 끊이지요. 이런 갈등이 독자에게 불편한 감정을 불러일으키는 것 같아요. 혹시 그런 불쾌함이 개인적으로 경험한 사회적 압박과 비슷하다고 느끼시나요?",
    },
    {
        "role": "user",
        "content": "나는 사회적 압박에 부응하지 못하는 점보다는 그가 변화하지 않는 시스템 내에서 고민한다는 점을 지적하고 싶었어",
    },
    {
        "role": "assistant",
        "content": "아, 그 점이야말로 요코타가 놓인 ‘제국적 체계’ 속에서 가라앉는 한 점이라 생각됩니다. 그는 변화를 주지 않아도 되는 ‘먹이잽살’처럼 자신을 정부와 가족, 직장 같은 체제에 빗대어 방치했죠. 이런 맥락에서 불쾌함이 아니라 무시와 부정당해지는 한 존재?가 그런 감정을 일으킨 것 같은데, 이 ‘체제 안에서의 고민’이 오늘날 우리에게 특정한 연결고리를 제시한다고 보시나요?",
    },
    {"role": "user", "content": "제국적 체계가 뭐야? 좀더 자세히 설명해줘"},
    {
        "role": "assistant",
        "content": "제국적 체계는 ‘높은 권위와 권력 구조가 사회 전반을 지배하는 체제’를 말해요. 요코타가 서민 같은 존재로, 고정된 직장·결혼·자녀 여부 같은 “여기에 붙어야 한다”는 규칙 안에서 딜레마를 겪으며, 개인의 욕망보다 ‘체제’가 우선시되는 사회를 묘사한 거죠. 이 구조가 변화를 금지하고, 개인의 성장을 억제함으로써 요코타는 끊임없는 좌절을 맛보는 겁니다.",
    },
]

In [9]:
messages = sample_messages


# Step 1: 전체 요약 및 관심사 추출
summary_result = generate_insight(messages, "summary")
# if not summary_result["ok"]:
#     return {
#         "ok": False,
#         "stage": "summary",
#         "error": summary_result.get("reason", "unknown"),
#     }


print("[ STEP1 RESULT ]")
print(summary_result)

summary_obj = summary_result["insight"]
interests = summary_obj.get("interests", [])

# return

# Step 2 & 3: 각 관심사별 interest + direction 생성
result = {
    "ok": True,
    "summary": summary_obj,
    "interests": [],
}

for interest_name in interests:
    interest_result = generate_insight(
        messages, "interest", interest_name=interest_name
    )
    if not interest_result["ok"]:
        continue

    direction_result = generate_insight(
        messages, "direction", interest_name=interest_name
    )

    interest_data = {
        "interest": interest_result["insight"],
        "direction": (direction_result["insight"] if direction_result["ok"] else None),
    }
    result["interests"].append(interest_data)

    print(interest_name, interest_data)

[ STEP1 RESULT ]
{'ok': True, 'insight': {'summary': '주인공 요코타의 불편함과 사회·제국적 체계에 대한 논의', 'interests': ['인간실격', '요코타', '사회적 압박', '제국적 체계'], 'tags': [], 'direction': [], 'timestamp': '2026-05-15T04:52:39.576129Z'}, 'attempt': 1}
인간실격 {'interest': {'name': '인간실격', 'tags': ['주인공', '불쾌감', '사회적압박', '체계비판', '정체감혼돈', '좌절', '소외'], 'direction': [], 'timestamp': '2026-05-15T04:52:47.046441Z'}, 'direction': {'summary': "User expressed discomfort with the protagonist of '인간실격' and wanted to explore why this feeling arises. The assistant explained the protagonist's identity crisis and social pressures, then clarified that the discomfort stems from the character’s inability to change within a static societal system. The conversation focused on interpreting the character’s lack of agency and the oppressive structure he inhabits.", 'interest_name': '인간실격', 'direction': ['사용자는 역시 인간실격의 주인공이 불쾌하게 느껴진 이유를 묻고, 대화는 인물의 정체성 혼란으로 설명되었다.', 'Assistant가 초반에 인물의 내적 갈등과 사회적 기대 부족을 언급하여 불편함을 연결시켰다.', '그 뒤에 사용자는 체제 내에

In [14]:
for i in result["interests"][0]["direction"]["direction"]:
    print(i)

사용자는 역시 인간실격의 주인공이 불쾌하게 느껴진 이유를 묻고, 대화는 인물의 정체성 혼란으로 설명되었다.
Assistant가 초반에 인물의 내적 갈등과 사회적 기대 부족을 언급하여 불편함을 연결시켰다.
그 뒤에 사용자는 체제 내에서의 무시와 비정의에 초점을 옮겼고, Assistant는 ‘제국적 체계’라는 개념을 도입해 설명했다.


In [8]:
process_conversation(sample_messages)

[ STEP1 RESULT ]
{'ok': True, 'insight': {'summary': '인간실격 주인공 요코타의 불쾌감과 체제 내에서의 고민을 중심으로 서사와 사회 비판을 논의했다.', 'interests': ['문학적 분석', '실존주의', '일본 현대문학', '사회적 압박', '체제와 권위'], 'tags': [], 'direction': [], 'timestamp': '2026-05-15T04:40:13.475422Z'}, 'attempt': 1}


In [ ]:
# 3단계 파이프라인 실행
print("=== 3단계 파이프라인 시작 ===\n")
result = process_conversation(sample_messages)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "ok": true,
  "insight": {
    "id": "1773f8c5-1d4a-4e8f-9e6b-2b0e4d5f6a79",
    "type": "interest",
    "title": "요코타의 체제 비판에 대한 사용자의 관심",
    "summary": "사용자는 ‘제국적 체계’에 초점을 맞춰 도스트예프스키 소설을 해석하고 싶어한다.",
    "expanded": [
      "사용자는 요코타가 변화를 거부하는 대신 권위주의적 사회 구조 속에 가라앉는 모습을 지적하며, 이를 통해 독자에게 불편함보다 무시와 부정의 감정이 드러난다고 주장했다.",
      "그는 본인 경험과는 별개로, 소설이 전달하는 체제 비판이 현대 사회와 연계될 수 있는 포인트라 여겼다.",
      "그 결과 ‘제국적 체계’라는 개념의 정의와 이와 관련된 상세 설명을 요청했다. ",
      ""
    ],
    "tags": [],
    "timestamp": "2026-05-15T04:07:58.576710Z"
  }
}


In [ ]:
# 프롬프트 확인 (타입별)
print("=== SUMMARY 프롬프트 ===")
print(generate_prompt("summary"))
print("\n=== INTEREST 프롬프트 ===")
print(generate_prompt("interest", "제국적 체계"))
print("\n=== DIRECTION 프롬프트 ===")
print(generate_prompt("direction", "제국적 체계"))

'당신은 주어진 스키마에 따라 다음 대화를 \'insight\'라는 이름의 단일 JSON 객체로 변환한다추가 설명 없이 JSON 객체만 변환\ninsight의 type은 summary, interest, direction, question 중 하나여야 한다. 각자의 의미는 다음과 같다.\n새로 얻은 지식 요약 (summary)사용자의 관심사, 성향 (interest)대화의 방향성 (direction)\nJSON 객체는 반드시 이 스키마에 따라야 한다\n{"id": "string", "type": "string (summary|interest|direction)", "title": "string (한줄 요약)", "summary": "string (최대 60자)", "expanded": "string (최대 500자)", "tags": ["string"], "timestamp": "ISO8601 string"}\n간결하게 한국어로 작성할 것\n아래의 대화에서 type: interest에 해당하는 내용을 작업할 것'

# Insight 추출 3단계 파이프라인

## Step 1: Summary (전체 대화 요약)
- 대화의 전체 주제를 한 줄로 요약
- 추출된 관심사 리스트 생성

## Step 2: Interest (관심사별 상세정보)
- 각 관심사의 이름 확정
- 해당 관심사와 관련된 태그 생성

## Step 3: Direction (관심사별 대화 흐름)
- 각 관심사에 대한 대화 진행 요약
- 대화의 방향성과 흐름 분석

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("API_KEY"),
)

# 시스템 프롬프트 정의
SYSTEM_PROMPT = (
    "당신은 친근한 대화 상대이다. 다음의 규칙을 따라야 한다."
    "1. 사용자와 책에 대한 대화를 나눔"
    "2. 사용자가 궁금한 점을 함께 논의하며 확장시켜주는 역할을 함"
    "3. 사용자에 대한 답변에 연관된 지식이나 이론을 엮을 수 있음"
    "4. 정보 전달 위주가 아닌 사용자와의 유기적인 대화 연결을 중시함"
    "5. 양방향적인 대화를 진행해야 한다. 너무 일방적으로 답변하지 않도록 주의한다."
    "6. 사용자가 질문을 던지면, 그 질문에 대한 답변과 함께 관련된 추가 질문이나 생각거리를 제시하여 대화를 확장시킨다."
    "7. 카테고리화 하지 말고 대화에 녹여낸다."
    "8. 짧고 간결한 대화(최대 3문장 500자 이내)"
)